# Phase 5: Feature Engineering

## 1. Introduction

Feature engineering is the process of creating and selecting useful input variables (features) from existing data.

In this phase, customer information from multiple datasets is combined to create a single customer-level dataset for machine learning.

## 2. Objectives

- Combine customer, subscription, payment, viewing, support, and feedback data.
- Create meaningful customer-level features.
- Handle missing values and duplicate records.
- Prepare the target variable for churn prediction.
- Avoid data leakage by excluding information that would not be available before a churn prediction.
- Export the final dataset for machine learning.

In [2]:
import pandas as pd
import numpy as np

# Load cleaned datasets
customers = pd.read_csv("/content/customers.csv")
subscriptions = pd.read_csv("/content/subscriptions.csv")
payments = pd.read_csv("/content/payments.csv")
viewing_activity = pd.read_csv("/content/viewing_activity.csv")
support_tickets = pd.read_csv("/content/support_tickets.csv")
customer_feedback = pd.read_csv("/content/customer_feedback.csv")
churn_labels = pd.read_csv("/content/churn_labels.csv")

print("All datasets loaded successfully!")

datasets = {
    "Customers": customers,
    "Subscriptions": subscriptions,
    "Payments": payments,
    "Viewing Activity": viewing_activity,
    "Support Tickets": support_tickets,
    "Customer Feedback": customer_feedback,
    "Churn Labels": churn_labels
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

All datasets loaded successfully!
Customers: (8000, 12)
Subscriptions: (8875, 13)
Payments: (110735, 7)
Viewing Activity: (66422, 11)
Support Tickets: (6120, 11)
Customer Feedback: (5046, 11)
Churn Labels: (8000, 4)


In [3]:
for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print(df.columns.tolist())


--- Customers ---
['customer_id', 'first_name', 'last_name', 'age', 'gender', 'country', 'city', 'registration_date', 'acquisition_channel', 'customer_segment', 'preferred_language', 'age_validation_flag']

--- Subscriptions ---
['subscription_id', 'customer_id', 'plan_id', 'subscription_start_date', 'subscription_end_date', 'subscription_status', 'auto_renew', 'cancellation_date', 'cancellation_reason', 'monthly_price', 'cancellation_date_flag', 'validated_cancellation_date', 'price_validation_flag']

--- Payments ---
['payment_id', 'customer_id', 'payment_date', 'amount', 'payment_method', 'payment_status', 'failed_payment_reason']

--- Viewing Activity ---
['viewing_id', 'customer_id', 'content_id', 'viewing_date', 'watch_duration_minutes', 'completion_percentage', 'device_type', 'login_location', 'session_duration_minutes', 'session_watch_gap', 'duration_mismatch_category']

--- Support Tickets ---
['ticket_id', 'customer_id', 'ticket_date', 'issue_category', 'issue_subcategory', 

In [4]:
# Check date ranges in all datasets
date_columns = {
    "Customers": (customers, "registration_date"),
    "Subscriptions": (subscriptions, "subscription_start_date"),
    "Payments": (payments, "payment_date"),
    "Viewing Activity": (viewing_activity, "viewing_date"),
    "Support Tickets": (support_tickets, "ticket_date"),
    "Customer Feedback": (customer_feedback, "feedback_date"),
    "Churn Labels": (churn_labels, "churn_date")
}

for name, (df, col) in date_columns.items():
    dates = pd.to_datetime(df[col], errors="coerce")
    print(f"\n{name}")
    print("Earliest date:", dates.min())
    print("Latest date:", dates.max())

print("\nChurn distribution:")
print(churn_labels["churned"].value_counts())

print("\nMissing churn labels:")
print(churn_labels["churned"].isna().sum())


Customers
Earliest date: 2019-01-01 00:00:00
Latest date: 2026-05-31 00:00:00

Subscriptions
Earliest date: 2019-01-01 00:00:00
Latest date: 2026-06-29 00:00:00

Payments
Earliest date: 2019-01-01 00:00:00
Latest date: 2026-05-31 00:00:00

Viewing Activity
Earliest date: 2025-07-01 00:00:00
Latest date: 2026-06-30 00:00:00

Support Tickets
Earliest date: 2019-03-03 00:00:00
Latest date: 2026-06-30 00:00:00

Customer Feedback
Earliest date: 2019-03-06 00:00:00
Latest date: 2026-06-30 00:00:00

Churn Labels
Earliest date: 2025-10-03 00:00:00
Latest date: 2026-06-30 00:00:00

Churn distribution:
churned
False    6009
True     1991
Name: count, dtype: int64

Missing churn labels:
0


In [5]:
# Convert date columns to datetime
date_columns = {
    "customers": ["registration_date"],
    "subscriptions": [
        "subscription_start_date",
        "subscription_end_date",
        "cancellation_date",
        "validated_cancellation_date"
    ],
    "payments": ["payment_date"],
    "viewing_activity": ["viewing_date"],
    "support_tickets": ["ticket_date"],
    "customer_feedback": ["feedback_date"],
    "churn_labels": ["churn_date"]
}

for name, columns in date_columns.items():
    df = globals()[name]
    for col in columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Create one reference date for each customer
reference_date = pd.Timestamp("2026-06-30")

customer_cutoffs = churn_labels[["customer_id", "churned", "churn_date"]].copy()

customer_cutoffs["feature_cutoff"] = customer_cutoffs["churn_date"].where(
    customer_cutoffs["churned"] == True,
    reference_date
)

customer_cutoffs["feature_cutoff"] = customer_cutoffs["feature_cutoff"].fillna(
    reference_date
)

print("Customer cutoffs created.")
print(customer_cutoffs.head())
print("\nCutoff date summary:")
print(customer_cutoffs["feature_cutoff"].describe())

Customer cutoffs created.
  customer_id  churned churn_date feature_cutoff
0  CUST000001    False        NaT     2026-06-30
1  CUST000002    False        NaT     2026-06-30
2  CUST000003     True 2026-04-20     2026-04-20
3  CUST000004    False        NaT     2026-06-30
4  CUST000005    False        NaT     2026-06-30

Cutoff date summary:
count                          8000
mean     2026-05-28 15:24:39.600000
min             2025-10-03 00:00:00
25%             2026-06-30 00:00:00
50%             2026-06-30 00:00:00
75%             2026-06-30 00:00:00
max             2026-06-30 00:00:00
Name: feature_cutoff, dtype: object


In [6]:
# Attach each customer's feature cutoff to payment records
payment_data = payments.merge(
    customer_cutoffs[["customer_id", "feature_cutoff"]],
    on="customer_id",
    how="inner"
)

# Keep only payments available before the customer's cutoff
payment_data = payment_data[
    payment_data["payment_date"] <= payment_data["feature_cutoff"]
].copy()

# Create a failed-payment indicator
payment_data["failed_payment"] = (
    payment_data["payment_status"].astype(str).str.lower() == "failed"
).astype(int)

# Aggregate payment information per customer
payment_features = payment_data.groupby("customer_id").agg(
    payment_count=("payment_id", "count"),
    failed_payment_count=("failed_payment", "sum"),
    average_payment_amount=("amount", "mean"),
    total_payment_amount=("amount", "sum")
).reset_index()

# Calculate payment failure rate
payment_features["payment_failure_rate"] = (
    payment_features["failed_payment_count"] /
    payment_features["payment_count"]
)

print("Payment features created!")
print("Shape:", payment_features.shape)
display(payment_features.head())
print("\nMissing values:")
print(payment_features.isnull().sum())

Payment features created!
Shape: (8000, 6)


,customer_id,payment_count,failed_payment_count,average_payment_amount,total_payment_amount,payment_failure_rate
0,CUST000001,15,0,22.223333,333.35,0.000000
1,CUST000002,6,0,15.490000,92.94,0.000000
2,CUST000003,15,0,22.990000,344.85,0.000000
3,CUST000004,15,1,6.990000,104.85,0.066667
4,CUST000005,15,1,6.990000,104.85,0.066667



Missing values:
customer_id               0
payment_count             0
failed_payment_count      0
average_payment_amount    0
total_payment_amount      0
payment_failure_rate      0
dtype: int64


In [7]:
# Attach customer cutoff dates to viewing records
viewing_data = viewing_activity.merge(
    customer_cutoffs[["customer_id", "feature_cutoff"]],
    on="customer_id",
    how="inner"
)

# Keep viewing activity before the customer's cutoff
viewing_data = viewing_data[
    viewing_data["viewing_date"] <= viewing_data["feature_cutoff"]
].copy()

# Aggregate viewing information per customer
viewing_features = viewing_data.groupby("customer_id").agg(
    viewing_sessions=("viewing_id", "count"),
    total_watch_minutes=("watch_duration_minutes", "sum"),
    average_watch_minutes=("watch_duration_minutes", "mean"),
    average_completion_percentage=("completion_percentage", "mean"),
    average_session_duration=("session_duration_minutes", "mean"),
    unique_content_watched=("content_id", "nunique"),
    unique_devices_used=("device_type", "nunique")
).reset_index()

print("Viewing features created!")
print("Shape:", viewing_features.shape)
display(viewing_features.head())

print("\nMissing values:")
print(viewing_features.isnull().sum())

Viewing features created!
Shape: (7848, 8)


,customer_id,viewing_sessions,total_watch_minutes,average_watch_minutes,average_completion_percentage,average_session_duration,unique_content_watched,unique_devices_used
0,CUST000001,5,226,45.200,77.20,57.200,5,3
1,CUST000002,5,206,41.200,65.60,53.200,5,3
2,CUST000003,2,84,42.000,76.50,48.500,2,1
3,CUST000004,8,471,58.875,63.75,66.125,8,5
4,CUST000005,10,397,39.700,70.00,50.000,10,5



Missing values:
customer_id                      0
viewing_sessions                 0
total_watch_minutes              0
average_watch_minutes            0
average_completion_percentage    6
average_session_duration         0
unique_content_watched           0
unique_devices_used              0
dtype: int64


## 4. Viewing Activity Feature Engineering

### Objective
To create customer-level viewing features that help measure customer engagement with the Netflix platform.

### Methodology
- Joined viewing activity with customer cutoff dates using `customer_id`.
- Included only viewing records dated on or before each customer's feature cutoff.
- Grouped viewing records by customer.
- Calculated viewing sessions, total and average watch time, average completion percentage, average session duration, unique content watched, and unique devices used.

### Key Findings
- Viewing features were successfully generated for **7,848 customers**.
- The resulting dataset contains **8 columns**, including the customer ID.
- Six missing values were found in `average_completion_percentage`.
- A total of 152 customers did not have viewing records available before their feature cutoff.

### Interpretation
Viewing sessions, watch time, completion percentage, and device usage help measure customer engagement. These features may help identify customers with lower platform engagement who could be at greater risk of churn.

### Conclusion
Viewing activity features were successfully created. Missing viewing records and completion values will be handled during the final dataset preparation stage.

In [8]:
# Attach customer cutoff dates to support tickets
support_data = support_tickets.merge(
    customer_cutoffs[["customer_id", "feature_cutoff"]],
    on="customer_id",
    how="inner"
)

# Keep support tickets before or on the customer's cutoff
support_data = support_data[
    support_data["ticket_date"] <= support_data["feature_cutoff"]
].copy()

# Aggregate support information per customer
support_features = support_data.groupby("customer_id").agg(
    support_ticket_count=("ticket_id", "count"),
    average_resolution_hours=("resolution_time_hours", "mean"),
    average_support_satisfaction=("customer_satisfaction_score", "mean"),
    unique_issue_categories=("issue_category", "nunique")
).reset_index()

print("Support features created!")
print("Shape:", support_features.shape)
display(support_features.head())

print("\nMissing values:")
print(support_features.isnull().sum())

Support features created!
Shape: (4229, 5)


,customer_id,support_ticket_count,average_resolution_hours,average_support_satisfaction,unique_issue_categories
0,CUST000001,2,10.9,2.5,2
1,CUST000002,1,6.4,3.0,1
2,CUST000003,1,NaN,NaN,1
3,CUST000007,1,2.4,3.0,1
4,CUST000010,1,58.6,5.0,1



Missing values:
customer_id                       0
support_ticket_count              0
average_resolution_hours        836
average_support_satisfaction    836
unique_issue_categories           0
dtype: int64


## 5. Support Ticket Feature Engineering

### Objective
To understand customer support interactions and identify whether support experiences may be associated with customer churn.

### Methodology
- Joined support ticket records with customer cutoff dates using `customer_id`.
- Included only tickets recorded on or before each customer's feature cutoff.
- Grouped support records by customer.
- Created the following features:
  - Support ticket count
  - Average resolution time
  - Average customer satisfaction score
  - Number of unique issue categories

### Key Findings
- Support features were generated for **4,229 customers**.
- The resulting dataset contains **5 columns**, including customer ID.
- There are 836 missing values in average resolution time.
- There are 836 missing values in average support satisfaction.
- The two missing-value counts may represent tickets without recorded resolution or satisfaction information.

### Interpretation
Frequent support interactions, longer resolution times, and lower satisfaction scores may indicate customer difficulties. These features can help the churn prediction model identify customers who may need additional support.

### Conclusion
Support ticket features were successfully created. Missing support measurements will be addressed during final dataset preparation.

In [9]:
# Attach customer cutoff dates to feedback records
feedback_data = customer_feedback.merge(
    customer_cutoffs[["customer_id", "feature_cutoff"]],
    on="customer_id",
    how="inner"
)

# Keep feedback submitted before or on the customer's cutoff
feedback_data = feedback_data[
    feedback_data["feedback_date"] <= feedback_data["feature_cutoff"]
].copy()

# Aggregate feedback information per customer
feedback_features = feedback_data.groupby("customer_id").agg(
    feedback_count=("feedback_id", "count"),
    average_feedback_rating=("rating", "mean"),
    negative_feedback_count=(
        "sentiment_label",
        lambda x: (x.astype(str).str.lower() == "negative").sum()
    ),
    positive_feedback_count=(
        "sentiment_label",
        lambda x: (x.astype(str).str.lower() == "positive").sum()
    )
).reset_index()

print("Feedback features created!")
print("Shape:", feedback_features.shape)
display(feedback_features.head())

print("\nMissing values:")
print(feedback_features.isnull().sum())

Feedback features created!
Shape: (3440, 5)


,customer_id,feedback_count,average_feedback_rating,negative_feedback_count,positive_feedback_count
0,CUST000001,2,3.0,1,1
1,CUST000002,1,1.0,1,0
2,CUST000003,1,3.0,0,0
3,CUST000008,1,3.0,0,0
4,CUST000009,1,4.0,0,1



Missing values:
customer_id                0
feedback_count             0
average_feedback_rating    0
negative_feedback_count    0
positive_feedback_count    0
dtype: int64


## 6. Customer Feedback Feature Engineering

### Objective
To capture customer opinions and satisfaction levels through feedback ratings and sentiment information.

### Methodology
- Joined customer feedback records with customer cutoff dates using `customer_id`.
- Included only feedback submitted on or before each customer's feature cutoff.
- Grouped feedback records by customer.
- Created the following features:
  - Total feedback count
  - Average feedback rating
  - Negative feedback count
  - Positive feedback count

### Key Findings
- Feedback features were successfully generated for **3,440 customers**.
- The resulting dataset contains **5 columns**, including customer ID.
- No missing values were found in the generated feedback features.

### Interpretation
Average ratings and positive or negative feedback counts help measure customer satisfaction. Customers providing negative feedback or consistently low ratings may require attention to reduce potential churn.

### Conclusion
Customer feedback features were successfully created and are ready to be combined with the other customer-level features.

In [10]:
# Keep useful customer information
customer_features = customers[[
    "customer_id",
    "age",
    "gender",
    "country",
    "acquisition_channel",
    "customer_segment",
    "preferred_language"
]].copy()

# Sort subscriptions by start date
subscription_data = subscriptions.sort_values(
    "subscription_start_date"
)

# Keep the latest subscription record for each customer
latest_subscriptions = subscription_data.drop_duplicates(
    subset="customer_id",
    keep="last"
)

# Select safe subscription features only
subscription_features = latest_subscriptions[[
    "customer_id",
    "plan_id",
    "subscription_start_date",
    "monthly_price",
    "auto_renew"
]].copy()

# Calculate subscription tenure at the customer's cutoff
subscription_features = subscription_features.merge(
    customer_cutoffs[["customer_id", "feature_cutoff"]],
    on="customer_id",
    how="left"
)

subscription_features["subscription_tenure_days"] = (
    subscription_features["feature_cutoff"] -
    subscription_features["subscription_start_date"]
).dt.days

# Remove helper cutoff column
subscription_features.drop(
    columns=["feature_cutoff"],
    inplace=True
)

print("Customer features shape:", customer_features.shape)
print("Subscription features shape:", subscription_features.shape)

print("\nCustomer features:")
display(customer_features.head())

print("\nSubscription features:")
display(subscription_features.head())

Customer features shape: (8000, 7)
Subscription features shape: (8000, 6)

Customer features:


,customer_id,age,gender,country,acquisition_channel,customer_segment,preferred_language
0,CUST000744,21.0,Female,United Kingdom,Organic Search,Individual,English
1,CUST001502,31.0,Male,Canada,Referral,Family,French
2,CUST005383,40.0,Male,United Kingdom,Paid Social,Student,English
3,CUST004708,27.0,Male,United Kingdom,Partner Bundle (Telecom),Individual,Hindi
4,CUST005877,36.0,Male,Indonesia,Email Campaign,Student,French



Subscription features:


,customer_id,plan_id,subscription_start_date,monthly_price,auto_renew,subscription_tenure_days
0,CUST002692,PLAN02,2019-01-01,15.49,True,2737
1,CUST003898,PLAN01,2019-01-01,6.99,True,2737
2,CUST006257,PLAN01,2019-01-01,6.99,True,2737
3,CUST004857,PLAN02,2019-01-01,15.49,False,2737
4,CUST007986,PLAN01,2019-01-01,6.99,True,2737


In [11]:
# Attach each customer's cutoff date to subscription records
subscription_data = subscriptions.merge(
    customer_cutoffs[["customer_id", "feature_cutoff"]],
    on="customer_id",
    how="inner"
)

# Keep only subscriptions that started on or before the cutoff
subscription_data = subscription_data[
    subscription_data["subscription_start_date"]
    <= subscription_data["feature_cutoff"]
].copy()

# Select the latest eligible subscription for each customer
latest_subscriptions = (
    subscription_data
    .sort_values("subscription_start_date")
    .drop_duplicates(subset="customer_id", keep="last")
)

# Select safe subscription features
subscription_features = latest_subscriptions[[
    "customer_id",
    "plan_id",
    "subscription_start_date",
    "monthly_price",
    "auto_renew"
]].copy()

# Calculate subscription tenure
subscription_features = subscription_features.merge(
    customer_cutoffs[["customer_id", "feature_cutoff"]],
    on="customer_id",
    how="left"
)

subscription_features["subscription_tenure_days"] = (
    subscription_features["feature_cutoff"]
    - subscription_features["subscription_start_date"]
).dt.days

subscription_features.drop(columns=["feature_cutoff"], inplace=True)

print("Corrected subscription features shape:", subscription_features.shape)
display(subscription_features.head())

print("\nNegative tenure values:",
      (subscription_features["subscription_tenure_days"] < 0).sum())

print("\nMissing values:")
print(subscription_features.isnull().sum())

Corrected subscription features shape: (8000, 6)


,customer_id,plan_id,subscription_start_date,monthly_price,auto_renew,subscription_tenure_days
0,CUST004857,PLAN02,2019-01-01,15.49,False,2737
1,CUST002692,PLAN02,2019-01-01,15.49,True,2737
2,CUST006257,PLAN01,2019-01-01,6.99,True,2737
3,CUST007986,PLAN01,2019-01-01,6.99,True,2737
4,CUST003898,PLAN01,2019-01-01,6.99,True,2737



Negative tenure values: 0

Missing values:
customer_id                 0
plan_id                     0
subscription_start_date     0
monthly_price               0
auto_renew                  0
subscription_tenure_days    0
dtype: int64


## 7. Customer and Subscription Feature Engineering

### Objective
To prepare customer demographic information and subscription characteristics for churn prediction.

### Methodology

**Customer features:**
- Selected customer age, gender, country, acquisition channel, customer segment, and preferred language.
- Retained one record per customer.

**Subscription features:**
- Selected the latest subscription that started on or before the customer's feature cutoff date.
- Extracted subscription plan, start date, monthly price, and auto-renew status.
- Calculated subscription tenure in days.
- Excluded cancellation-related fields to prevent data leakage.

### Key Findings
- Customer features contain 8,000 records and 7 columns.
- Subscription features contain 8,000 records and 6 columns.
- No missing values were found in the corrected subscription features.
- No negative subscription tenure values were found.
- All selected subscriptions started on or before the respective customer cutoff dates.

### Interpretation
Customer demographics and subscription characteristics provide useful information about customer profiles, pricing, and subscription behaviour. Subscription tenure and auto-renew status may help identify patterns associated with churn.

### Conclusion
Customer and subscription features were successfully prepared and validated for integration with the payment, viewing, support, and feedback features.

In [12]:
# Start with the complete customer dataset
final_features = customer_features.copy()

# Merge subscription features
final_features = final_features.merge(
    subscription_features,
    on="customer_id",
    how="left"
)

# Merge payment features
final_features = final_features.merge(
    payment_features,
    on="customer_id",
    how="left"
)

# Merge viewing features
final_features = final_features.merge(
    viewing_features,
    on="customer_id",
    how="left"
)

# Merge support features
final_features = final_features.merge(
    support_features,
    on="customer_id",
    how="left"
)

# Merge feedback features
final_features = final_features.merge(
    feedback_features,
    on="customer_id",
    how="left"
)

# Add churn target
final_features = final_features.merge(
    churn_labels[["customer_id", "churned"]],
    on="customer_id",
    how="left"
)

print("Final merged dataset shape:", final_features.shape)

print("\nFirst five rows:")
display(final_features.head())

print("\nMissing values:")
print(final_features.isnull().sum())

Final merged dataset shape: (8000, 33)

First five rows:


,customer_id,age,gender,country,acquisition_channel,customer_segment,preferred_language,plan_id,subscription_start_date,monthly_price,...,unique_devices_used,support_ticket_count,average_resolution_hours,average_support_satisfaction,unique_issue_categories,feedback_count,average_feedback_rating,negative_feedback_count,positive_feedback_count,churned
0,CUST000744,21.0,Female,United Kingdom,Organic Search,Individual,English,PLAN03,2019-06-03,22.99,...,4.0,NaN,NaN,NaN,NaN,1.0,3.0,0.0,0.0,False
1,CUST001502,31.0,Male,Canada,Referral,Family,French,PLAN01,2026-01-13,6.99,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,CUST005383,40.0,Male,United Kingdom,Paid Social,Student,English,PLAN01,2023-06-17,6.99,...,4.0,NaN,NaN,NaN,NaN,1.0,4.0,0.0,1.0,False
3,CUST004708,27.0,Male,United Kingdom,Partner Bundle (Telecom),Individual,Hindi,PLAN02,2019-03-09,15.49,...,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4,CUST005877,36.0,Male,Indonesia,Email Campaign,Student,French,PLAN03,2025-12-03,22.99,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False



Missing values:
customer_id                         0
age                               329
gender                              0
country                           409
acquisition_channel                 0
customer_segment                    0
preferred_language                251
plan_id                             0
subscription_start_date             0
monthly_price                       0
auto_renew                          0
subscription_tenure_days            0
payment_count                       0
failed_payment_count                0
average_payment_amount              0
total_payment_amount                0
payment_failure_rate                0
viewing_sessions                  152
total_watch_minutes               152
average_watch_minutes             152
average_completion_percentage     158
average_session_duration          152
unique_content_watched            152
unique_devices_used               152
support_ticket_count             3771
average_resolution_hours         

In [13]:
# Create indicators before filling missing values
final_features["has_viewing_activity"] = (
    final_features["viewing_sessions"].notna().astype(int)
)

final_features["has_support_tickets"] = (
    final_features["support_ticket_count"].notna().astype(int)
)

final_features["has_feedback"] = (
    final_features["feedback_count"].notna().astype(int)
)

# Fill viewing count and total features with zero
viewing_zero_columns = [
    "viewing_sessions",
    "total_watch_minutes",
    "unique_content_watched",
    "unique_devices_used"
]

final_features[viewing_zero_columns] = (
    final_features[viewing_zero_columns].fillna(0)
)

# Fill support count features with zero
support_zero_columns = [
    "support_ticket_count",
    "unique_issue_categories"
]

final_features[support_zero_columns] = (
    final_features[support_zero_columns].fillna(0)
)

# Fill feedback count features with zero
feedback_zero_columns = [
    "feedback_count",
    "negative_feedback_count",
    "positive_feedback_count"
]

final_features[feedback_zero_columns] = (
    final_features[feedback_zero_columns].fillna(0)
)

# Fill numerical columns with their median
numerical_columns = final_features.select_dtypes(
    include=["number"]
).columns.tolist()

numerical_columns = [
    col for col in numerical_columns if col != "churned"
]

for col in numerical_columns:
    final_features[col] = final_features[col].fillna(
        final_features[col].median()
    )

# Fill categorical missing values
categorical_columns = final_features.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_columns:
    final_features[col] = final_features[col].fillna("Unknown")

# Convert churn target to integer
final_features["churned"] = (
    final_features["churned"].astype(int)
)

print("Missing value treatment completed!")

print("\nRemaining missing values:")
print(final_features.isnull().sum().sum())

print("\nFinal dataset shape:", final_features.shape)

print("\nChurn distribution:")
print(final_features["churned"].value_counts())

Missing value treatment completed!

Remaining missing values:
0

Final dataset shape: (8000, 36)

Churn distribution:
churned
0    6009
1    1991
Name: count, dtype: int64


## 8. Missing Value Treatment

### Objective
To handle missing values in the merged customer-level dataset while preserving all customer records.

### Methodology
- Created indicators to identify customers with viewing activity, support tickets, and feedback records.
- Replaced missing viewing, support, and feedback count features with zero where no corresponding records existed.
- Used median imputation for missing numerical measurements.
- Replaced missing categorical values with `Unknown`.
- Converted the churn target into integer format (0 and 1).

### Key Findings
- The dataset initially contained missing values across demographic, viewing, support, and feedback features.
- After treatment, all missing values were successfully handled.
- The final dataset contains **8,000 records and 36 columns**.
- Remaining missing values: **0**.
- Non-churned customers: **6,009**.
- Churned customers: **1,991**.

### Interpretation
Missing-value treatment preserved all 8,000 customers instead of removing incomplete records. Additional activity indicators distinguish customers with no recorded activity from customers who have actual activity measurements.

### Conclusion
Missing values were successfully handled, and the dataset is ready for final validation before machine learning preparation.

In [14]:
# Final data quality checks
print("DATA QUALITY CHECKS")

print("\n1. Dataset shape:", final_features.shape)

print("\n2. Duplicate customer IDs:",
      final_features["customer_id"].duplicated().sum())

print("\n3. Total missing values:",
      final_features.isnull().sum().sum())

print("\n4. Target values:",
      sorted(final_features["churned"].unique()))

print("\n5. Target distribution:")
print(final_features["churned"].value_counts())

print("\n6. Potential leakage columns:")
leakage_keywords = [
    "churn_date",
    "churn_reason",
    "cancellation_date",
    "cancellation_reason",
    "subscription_status"
]

print([
    col for col in final_features.columns
    if any(word in col.lower() for word in leakage_keywords)
])

print("\n7. Data types:")
print(final_features.dtypes)

DATA QUALITY CHECKS

1. Dataset shape: (8000, 36)

2. Duplicate customer IDs: 0

3. Total missing values: 0

4. Target values: [np.int64(0), np.int64(1)]

5. Target distribution:
churned
0    6009
1    1991
Name: count, dtype: int64

6. Potential leakage columns:
[]

7. Data types:
customer_id                              object
age                                     float64
gender                                   object
country                                  object
acquisition_channel                      object
customer_segment                         object
preferred_language                       object
plan_id                                  object
subscription_start_date          datetime64[ns]
monthly_price                           float64
auto_renew                                 bool
subscription_tenure_days                  int64
payment_count                             int64
failed_payment_count                      int64
average_payment_amount                  float

## 9. Final Data Quality and Leakage Checks

### Objective
To validate the final customer-level dataset and ensure it is suitable for machine learning.

### Validation Performed
- Checked the final dataset dimensions.
- Verified customer ID uniqueness.
- Checked for missing values.
- Validated the binary churn target.
- Checked for direct churn and cancellation-related columns.
- Reviewed the data types of all features.

### Key Findings
- Final dataset size: **8,000 rows and 36 columns**.
- Duplicate customer IDs: **0**.
- Missing values: **0**.
- Target values: 0 (Not Churned) and 1 (Churned).
- Direct churn or cancellation leakage columns detected: **None**.
- The dataset contains numerical, categorical, Boolean, and date features.

### Interpretation
The dataset passed the main structural and completeness checks. Customer ID is retained for identification but must be excluded from model training. The subscription start date will also require appropriate treatment before model training.

The feature aggregates were restricted to each customer's cutoff date. Since records on the exact churn date were included, same-day event timing remains

In [15]:
# Export the final feature-engineered dataset
output_path = "/content/netflix_feature_engineered.csv"

final_features.to_csv(output_path, index=False)

print("Feature-engineered dataset exported successfully!")
print("File:", output_path)
print("Rows:", final_features.shape[0])
print("Columns:", final_features.shape[1])

Feature-engineered dataset exported successfully!
File: /content/netflix_feature_engineered.csv
Rows: 8000
Columns: 36


## 10. Overall Conclusion

The feature engineering phase successfully combined seven cleaned datasets into a single customer-level dataset for Netflix churn prediction.

Customer demographic, subscription, payment, viewing activity, support interaction, and feedback information were transformed into meaningful machine-learning features.

The final dataset contains 8,000 customer records and 36 columns, including the churn target. Missing values were handled, duplicate customer IDs were not found, and the final validation checks were completed.

The dataset was exported as `netflix_feature_engineered.csv` and is ready for the next phase: **Churn Prediction using Machine Learning**.